# Tools 🔨🪚🔩⚙️⚒️🧲🧰🔧🪛🔩⚙️🔦🧭🔗⛓️🧮
Tools allow agents to 'Act' in the real world. Careful descriptions can help your agent discover how to use your tools.

LangChain supports many tool formats and tool sets. Here we will cover some common cases, but check the docs for more information.

## Calculator example
In this example, the docstring and inferred arguments and argument types are used by the LLM to detetermine when and how to call the tool.

In [3]:
from typing import Literal
from langchain.tools import tool

@tool
def real_number_calculator(
    a: float, b: float, operation: Literal["add", "subtract", "multiply", "divide"] 
) -> float:
    """Perform basic arithmetic operations on two real numbers."""
    print("🧮 Invoking calculator tool")
    # Perform the specified operation
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            raise ValueError("Division by zero is not allowed.")
        return a / b
    else:
        raise ValueError(f"Invalid operation: {operation}")


In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model = "ollama:qwen2.5:3b",
    tools = [real_number_calculator],
    system_prompt="You are a helpful assistant",
)

This invokes your calculator tool.

In [5]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "what is 12.55 * 13.66"}]}
)
print(result["messages"][-1].content)

🧮 Invoking calculator tool
The result of multiplying 12.55 by 13.66 is approximately 171.433.



We can check the metadata in LangSmith Observability to see this.

The tool description can have a big impact. This may not invoke your calculator tool because the inputs are integers. (results vary from run to run)

In [6]:
result = agent.invoke({"messages": [{"role": "user", "content": "what is 3 * 4"}]})
print(result["messages"][-1].content)

🧮 Invoking calculator tool
The result of multiplying 3 by 4 is 12.0.


This often does not invoke the tool though the input are real numbers. (results vary from run to run)

In [7]:
result = agent.invoke({"messages": [{"role": "user", "content": "what is 3.0 * 4.0"}]})
print(result["messages"][-1].content)

🧮 Invoking calculator tool
The result of multiplying 3.0 by 4.0 is 12.0.


## Adding a more detailed description
While a basic description is often sufficient, LangChain has support for enhanced descriptions. The example below uses one method: Google Style argument descriptions. Used with parse_docstring=True, this will parse and pass the arg descriptions to the model. You can rename the tool and change its description. This can be effective when you are sharing a standard tool but would like agent-specific instructions.

In [8]:
from typing import Literal

from langchain.tools import tool


@tool(
    "calculator",
    parse_docstring=True,
    description=(
        "Perform basic arithmetic operations on two real numbers."
        "Use this whenever you have operations on any numbers, even if they are integers."
    ),
)
def real_number_calculator(
    a: float, b: float, operation: Literal["add", "subtract", "multiply", "divide"]
) -> float:
    """Perform basic arithmetic operations on two real numbers.

    Args:
        a (float): The first number.
        b (float): The second number.
        operation (Literal["add", "subtract", "multiply", "divide"]):
            The arithmetic operation to perform.

            - `"add"`: Returns the sum of `a` and `b`.
            - `"subtract"`: Returns the result of `a - b`.
            - `"multiply"`: Returns the product of `a` and `b`.
            - `"divide"`: Returns the result of `a / b`. Raises an error if `b` is zero.

    Returns:
        float: The numerical result of the specified operation.

    Raises:
        ValueError: If an invalid operation is provided or division by zero is attempted.
    """
    print("🧮  Invoking calculator tool")
    # Perform the specified operation
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            raise ValueError("Division by zero is not allowed.")
        return a / b
    else:
        raise ValueError(f"Invalid operation: {operation}")

In [10]:
from langchain.agents import create_agent

agent = create_agent(
    model = "ollama:qwen2.5:3b",
    tools = [real_number_calculator],
    system_prompt="You are a helpful assistant",
)

In [11]:
result = agent.invoke({"messages": [{"role": "user", "content": "what is 3.0 * 4.0"}]})
print(result["messages"][-1].content)

🧮  Invoking calculator tool
The result of multiplying 3.0 by 4.0 is 12.0.


In [12]:
result = agent.invoke({"messages": [{"role": "user", "content": "what is 3 * 4"}]})
print(result["messages"][-1].content)

🧮  Invoking calculator tool
The result of multiplying 3 by 4 is 12.0.
